# Spektrum FFT + Logistic Regression
        **timeseries › classification_gunpoint** · notebook `fft_logreg`

        **Kapan dipakai:** Pola berbeda pada frekuensi (getaran, sinyal sensor, EKG).

        **Jebakan umum:** Mengabaikan fase; kurang cocok untuk pola yang dibedakan oleh bentuk.

**Data demo:** GunPoint (UCR). Tiap baris = satu deret, kolom `t0..tN` = nilai pada tiap waktu, `label` = kelas.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/timeseries/classification/gunpoint_train.csv"; TEST_PATH = ROOT / "data_latihan/timeseries/classification/gunpoint_test.csv"      # test berlabel (untuk evaluasi akhir)
LABEL_COL  = "label"
N_SPLITS, SEED = 5, 42
MODEL_KEY  = "fft_logreg"
OUT = out_dir("timeseries", "classification_gunpoint", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
tr, te = pd.read_csv(TRAIN_PATH), pd.read_csv(TEST_PATH)
feat = [c for c in tr.columns if c != LABEL_COL]
Xtr, ytr, Xte, yte = tr[feat].to_numpy(float), tr[LABEL_COL].astype(str).to_numpy(), te[feat].to_numpy(float), te[LABEL_COL].astype(str).to_numpy()
print(Xtr.shape, Xte.shape, "| panjang deret:", Xtr.shape[1], "| kelas:", dict(pd.Series(ytr).value_counts()))
fig, ax = plt.subplots(figsize=(8, 3))
for c in sorted(set(ytr)):
    m_, s_ = Xtr[ytr == c].mean(0), Xtr[ytr == c].std(0); ax.plot(m_, label=f"kelas {c}"); ax.fill_between(range(len(m_)), m_ - s_, m_ + s_, alpha=.15)
ax.legend(); ax.set_title("Rata-rata ± std per kelas"); plt.tight_layout(); plt.show()

## 3. Model

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from superdb.timeseries import ZNorm, ts_stat_features, TorchTSClassifier
from superdb.tsfeatures import fft_features, TsfreshFeatures, RocketTransform, DTWKNN
from sklearn.linear_model import LogisticRegression

model = make_pipeline(FunctionTransformer(fft_features), StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))

## 4. Cross-validation pada training

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import time
t0 = time.time()
cv = cross_val_score(model, Xtr, ytr, cv=StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED), scoring="f1_macro")
print(f"CV f1_macro = {cv.mean():.4f} ± {cv.std():.4f}  ({time.time() - t0:.0f} dtk)")

## 5. Latih & evaluasi pada test

In [ ]:
t0 = time.time(); model.fit(Xtr, ytr); fit_sec = time.time() - t0
pred = model.predict(Xte); classes = sorted(set(ytr))
proba = model.predict_proba(Xte) if hasattr(model, "predict_proba") else None
yi, pi = np.searchsorted(classes, yte), np.searchsorted(classes, pred)
metrics = classification_metrics(yi, pi, proba, classes=np.arange(len(classes))); display(show_table(metrics))
print(f"waktu latih: {fit_sec:.1f} dtk")
plot_confusion(yi, pi, labels=np.arange(len(classes)), path=OUT / "confusion.png"); plt.show()

## 6. Deret yang salah diprediksi

In [ ]:
wrong = np.where(pred != yte)[0][:4]
if len(wrong):
    fig, ax = plt.subplots(1, len(wrong), figsize=(3.2 * len(wrong), 2.6), squeeze=False)
    for a, i in zip(ax[0], wrong): a.plot(Xte[i]); a.set_title(f"benar {yte[i]} / prediksi {pred[i]}", fontsize=9)
    plt.tight_layout(); plt.savefig(OUT / "errors.png", dpi=130); plt.show()
save_json({"cv_f1_macro_mean": float(cv.mean()), "cv_f1_macro_std": float(cv.std()), "test": metrics, "fit_seconds": fit_sec}, OUT / "metrics.json")
save_model(model, OUT / "model.joblib"); b = load_model(OUT / "model.joblib"); assert (b.predict(Xte[:20]) == model.predict(Xte[:20])).all(); print("OK. hasil di", OUT)

## Langkah berikutnya
- Data sedikit (puluhan deret): ROCKET/MiniRocket dan 1-NN DTW biasanya terbaik; deep learning butuh data lebih banyak.
- Selalu normalisasi per deret bila amplitudo tidak informatif (sudah di pipeline).